In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install transformers torch scikit-learn tqdm

In [11]:
from transformers import AutoTokenizer, AutoModel
import torch
import os
import numpy as np
from tqdm import tqdm
from torch.nn.functional import cosine_similarity

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()


def encode(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.last_hidden_state[:, 0, :]


def load_cases(folder_path):
    cases = {}
    for file in os.listdir(folder_path):
        if file.endswith(".txt"):
            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
                cases[file] = f.read()
    return cases


def precompute_embeddings(cases):
    embeddings = {}
    for cid, text in tqdm(cases.items(), desc="Encoding cases"):
        embeddings[cid] = encode(text)
    return embeddings


def fast_rank(query_id, embeddings):
    query_vec = embeddings[query_id]

    scores = []
    for cid, vec in embeddings.items():
        if cid == query_id:
            continue
        score = cosine_similarity(query_vec, vec).item()
        scores.append((cid, score))

    ranked = sorted(scores, key=lambda x: x[1], reverse=True)
    return [cid for cid, _ in ranked]


def _to_float(x):
    try:
        return float(x)
    except Exception:
        return x


def compute_ir_metrics(top_k, relevant_set, topk):
    tp = len(set(top_k) & relevant_set)

    # Precision / Recall / F1
    p = tp / topk
    r = tp / len(relevant_set) if relevant_set else 0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0

    # MRR
    rr = 0
    for i, doc in enumerate(top_k):
        if doc in relevant_set:
            rr = 1 / (i + 1)
            break

    # NDCG
    dcg = 0
    for i, doc in enumerate(top_k):
        if doc in relevant_set:
            dcg += 1 / np.log2(i + 2)

    ideal_hits = min(len(relevant_set), topk)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits)) if ideal_hits > 0 else 0
    ndcg = dcg / idcg if idcg > 0 else 0

    # MAP
    hits = 0
    ap = 0
    for i, doc in enumerate(top_k):
        if doc in relevant_set:
            hits += 1
            ap += hits / (i + 1)

    ap = ap / len(relevant_set) if relevant_set else 0

    return tp, p, r, f, rr, ndcg, ap


def evaluate_with_yf(
    cases,
    ground_truth,
    embeddings,
    candidate_dict=None,
    topk=5,
    log_path=None,
    method_name="LEGAL-BERT"
):
    # NORMAL
    correct_pred = retri_cases = relevant_cases = 0
    macro_p = macro_r = macro_f = 0
    mrr_list, ndcg_list, ap_list = [], [], []

    # YF
    correct_pred_yf = retri_cases_yf = relevant_cases_yf = 0
    macro_p_yf = macro_r_yf = macro_f_yf = 0
    mrr_yf_list, ndcg_yf_list, ap_yf_list = [], [], []

    num_queries = 0
    missing_queries = []
    missing_docs = set()

    for query_id, relevant_list in ground_truth.items():

        if query_id not in cases:
            missing_queries.append(query_id)
            continue

        for rel in relevant_list:
            if rel not in cases:
                missing_docs.add(rel)

        ranked_full = fast_rank(query_id, embeddings)
        top_k = ranked_full[:topk]

        relevant_set = set(relevant_list)

        # NORMAL
        tp, p, r, f, rr, ndcg, ap = compute_ir_metrics(top_k, relevant_set, topk)

        correct_pred += tp
        retri_cases += topk
        relevant_cases += len(relevant_set)

        macro_p += p
        macro_r += r
        macro_f += f

        mrr_list.append(rr)
        ndcg_list.append(ndcg)
        ap_list.append(ap)

        # YF
        if candidate_dict:
            cand_set = set(candidate_dict[query_id])
            ranked_yf = [c for c in ranked_full if c in cand_set]
            top_k_yf = ranked_yf[:topk]

            tp_yf, p_yf, r_yf, f_yf, rr_yf, ndcg_yf, ap_yf = compute_ir_metrics(
                top_k_yf, relevant_set, topk
            )

            correct_pred_yf += tp_yf
            retri_cases_yf += topk
            relevant_cases_yf += len(relevant_set)

            macro_p_yf += p_yf
            macro_r_yf += r_yf
            macro_f_yf += f_yf

            mrr_yf_list.append(rr_yf)
            ndcg_yf_list.append(ndcg_yf)
            ap_yf_list.append(ap_yf)

        num_queries += 1

    # FINAL NORMAL
    micro_pre = correct_pred / retri_cases if retri_cases else 0
    micro_recall = correct_pred / relevant_cases if relevant_cases else 0
    micro_f = 2 * micro_pre * micro_recall / (micro_pre + micro_recall) if (micro_pre + micro_recall) > 0 else 0

    macro_pre = macro_p / num_queries if num_queries else 0
    macro_recall = macro_r / num_queries if num_queries else 0
    macro_f = macro_f / num_queries if num_queries else 0

    mrr = np.mean(mrr_list) if mrr_list else 0
    ndcg = np.mean(ndcg_list) if ndcg_list else 0
    map_score = np.mean(ap_list) if ap_list else 0

    # FINAL YF
    if candidate_dict:
        micro_pre_yf = correct_pred_yf / retri_cases_yf if retri_cases_yf else 0
        micro_recall_yf = correct_pred_yf / relevant_cases_yf if relevant_cases_yf else 0
        micro_f_yf = 2 * micro_pre_yf * micro_recall_yf / (micro_pre_yf + micro_recall_yf) if (micro_pre_yf + micro_recall_yf) > 0 else 0

        macro_pre_yf = macro_p_yf / num_queries if num_queries else 0
        macro_recall_yf = macro_r_yf / num_queries if num_queries else 0
        macro_f_yf = macro_f_yf / num_queries if num_queries else 0

        mrr_yf = np.mean(mrr_yf_list) if mrr_yf_list else 0
        ndcg_yf = np.mean(ndcg_yf_list) if ndcg_yf_list else 0
        map_yf = np.mean(ap_yf_list) if ap_yf_list else 0
    else:
        micro_pre_yf = micro_recall_yf = micro_f_yf = 0
        macro_pre_yf = macro_recall_yf = macro_f_yf = 0
        mrr_yf = ndcg_yf = map_yf = 0

    # LOGGING
    log_lines = [
        f"Method: {method_name}",
        f"TopK: {topk}",
        f"Queries: {num_queries}",

        f"NDCG@{topk}: {_to_float(ndcg)}",
        f"MRR@{topk}: {_to_float(mrr)}",
        f"MAP: {_to_float(map_score)}",
    ]

    if candidate_dict:
        log_lines += [
            f"NDCG@{topk} yf: {_to_float(ndcg_yf)}",
            f"MRR@{topk} yf: {_to_float(mrr_yf)}",
            f"MAP yf: {_to_float(map_yf)}",
        ]

    log_lines.append("")

    print("\n".join(log_lines))

    if log_path:
        with open(log_path, "w", encoding="utf-8") as f:
            f.write("\n".join(log_lines))
        print(f"Log saved to: {log_path}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
import json
# Load data
cases = load_cases("/content/drive/MyDrive/CaseGNNDataset/task1_test_2017/summary_test_2017_txt/")
labels = json.load(open("/content/drive/MyDrive/CaseGNNDataset/task1_test_2017/task1_test_labels_2017.json"))
candidate_dict = json.load(open("/content/drive/MyDrive/CaseGNNDataset/task1_test_2017/test_2017_candidate_with_yearfilter.json"))

In [13]:
# Precompute embeddings
embeddings = precompute_embeddings(cases)

Encoding cases: 100%|██████████| 168/168 [00:03<00:00, 46.16it/s]


In [15]:
evaluate_with_yf(
    cases=cases,
    ground_truth=labels,
    embeddings=embeddings,
    candidate_dict=candidate_dict,
    topk=5,
    log_path="/content/legalbert.log"
)

Method: LEGAL-BERT
TopK: 5
Queries: 38
NDCG@5: 0.08725332196643339
MRR@5: 0.13859649122807016
MAP: 0.05486842105263158
NDCG@5 yf: 0.12187960424635491
MRR@5 yf: 0.18859649122807015
MAP yf: 0.08451754385964913

Log saved to: /content/legalbert.log
